# Theoretical spectrum and significance spectrums

**Aim:** test whether the spectra, coherence or phase observed at a particular position on the timescale plane is not due to a random process

**How:** One starts with the observed time series, which are to be tested against the hypothesis of control datasets (synthetic or surrogate data), which share some statistical properties with the original series, but are generated by random processes.

### In the context of coherence

My question is: At this time & scale, are two signals related, beyond what we'd expect by chance?

**NULL HYPOTHESIS:**
These two time series are unrelated stochastic processes.

For example Torrence and Combo use two AR(1) processes, although in my case this might be too weak:

Real data (like EVI + prec):
- have strong seasonality
- share spectral peaks (annual cycle)
- are nonstationary

I might detect high coherence simply because they have same periodicities, not because they are one depend on the other. So what I really need to answer is: "What level of coherence would I expect if these signals had the same individual structure, but were NOT related?"

So I must preserve:
- each series' autocorrelation / spectrum
- but destroy cross-dependence

## AR(1) - Classical Torrence and Combo
Autoregressive process with red noise characteristics is defined by 

$y_t = \alpha y_{t-1}+ \epsilon_t$ 

where $\alpha$ is the lag-1 autocorrelation coefficient and $\epsilon_t$ is taken from gaussian white noise (new randomness entering the system) and $y_0=0$. 

following Cazelles et al 2014 we want $\alpha$ to be generated using the same autocorrelation coefficient as the EVI signal ($x_t$). To do so we can calculate it as:
$$
\alpha = \frac{\sum_{t=2}^{N}(x_t - \bar{x})(x_{t-1} - \bar{x})}{\sum_{t=1}^{N}(x_t - \bar{x})^2}
$$

this might be helpful: https://www.learnbymarketing.com/definitions/autocorrelation/
or this one: https://www.interactivebrokers.com/campus/ibkr-quant-news/autocorrelation-and-autocovariance-calculation-examples-and-more-part-i/ 


We are estimating:

“How much does the signal remember its previous value?”

 - α≈0 → no memory (white noise)
 - α≈1 → very persistent

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wavelets_wrapper_posh.quick_wavelet import run_full_wavelet_analysis, run_double_wavelet_analysis, unmirror_wavelet_power, unmirror_crosspower, unmirror_coherence, unmirror_phase


In [ ]:
# pre-process signal and remove mean, trend and seasonality
def preprocess_signal(x, deseason_period=None, detrend=True, demean=True, dt=16):
    x = np.array(x)

    # remove seasonal cycle
    if deseason_period is None:
        print("No seasonal cycle removed.")
    else:
        num_samples = int(round(deseason_period / dt))
        climatology = np.array([np.mean(x[i::num_samples]) for i in range(num_samples)])
        seasonal = np.tile(climatology, len(x)//num_samples + 1)[:len(x)]
        x = x - seasonal
        # #   plot for troubleshooting
        # plt.figure(figsize=(10, 4))
        # plt.plot(x, label='Deseasoned Signal')
        # plt.legend()
        # plt.title("Deseasoning the signal")
        # plt.show()
    # remove linear trend
    if detrend:
        t = np.arange(len(x))
        coeffs = np.polyfit(t, x, 1)
        trend = np.polyval(coeffs, t)
        x = x - trend
        # # plot for troubleshooting
        # plt.figure(figsize=(10, 4))
        # plt.plot(x, label='Detrended Signal')
        # plt.legend()
        # plt.title("Detrending the signal")
        # plt.show()
    else:
        print("No linear trend removed.")

    # remove mean (after deseasoning the signal might not be zero mean anymore)
    if demean:
        x = x - np.mean(x)
        # # plot for troubleshooting
        # plt.figure(figsize=(10, 4))
        # plt.plot(x, label='Demeaned Signal')
        # plt.legend()
        # plt.title("Demeaning the signal")
        # plt.show()
    else:
        print("No mean removed.")

    return x

# estimate alpha
def estimate_alpha(x, deseason_period=None):
    x_proc = preprocess_signal(x, deseason_period=deseason_period)
    alpha = np.corrcoef(x_proc[:-1], x_proc[1:])[0, 1] # calculate alpha from two shifted version of the signal
    return alpha

def generate_ar1(x, n=None, alpha=None, deseason_period=None):
    if n is None:
        n = len(x)

    # preprocess once
    x_proc = preprocess_signal(x, deseason_period=deseason_period)

    if alpha is None:
        alpha = np.corrcoef(x_proc[:-1], x_proc[1:])[0, 1]
    else:
        print(f"Using provided alpha: {alpha:.2f}")

    var = np.var(x_proc)
    sigma_eps = np.sqrt(var * (1 - alpha**2))

    # spin-up included (important!)
    y = np.zeros(n + 500)
    for t in range(1, len(y)):
        y[t] = alpha * y[t-1] + np.random.normal(0, sigma_eps)

    y = y[500:]  # remove spin-up

    return y, alpha

# make a function that given a signal it generates 1000 AR(1)
def generate_ar1_ensemble(x, n_ensemble=1000, n=None, alpha=None, deseason_period=None):
    ar1_ensemble = np.zeros((n_ensemble, n if n is not None else len(x)))
    
    for i in range(n_ensemble):
        ar1_ensemble[i], _ = generate_ar1(x, n=n, alpha=alpha, deseason_period=deseason_period)
    
    return ar1_ensemble

In [ ]:
# Aggregate across the ensemble 
def quantile_stats(grp):
    return grp.agg(
        median='median',
        p95=lambda s: s.quantile(0.95),
        p75=lambda s: s.quantile(0.75),
        p25=lambda s: s.quantile(0.25),
        )

In [ ]:
# generate a function that run a wavelet analysis for the 1000 AR(1) series and return the 95% percentile of the distribution
# my wavelet power will be significant if the value of the wavelet tranform is above this value

def estimate_significance_threshold(x, n_ensemble=1000, n=None, alpha=None, max_period=None, dt=16, deseason_period=None):

    """
    Estimate significance thresholds for wavelet power using an ensemble of
    AR(1) surrogate time series.

    An ensemble of AR(1) series is generated from the input time series, and
    a wavelet analysis is performed on each realisation. The distribution of
    wavelet power at each time and period is then used to calculate quantile
    statistics. The 95th percentile can be used as the significance threshold,
    so that observed wavelet power values above this level are considered
    significant.

    Parameters
    ----------
    x : array-like
        Input time series used to estimate the AR(1) coefficient and set the
        length of the surrogate series.
    n_ensemble : int
        Number of AR(1) realisations. Default is 1000.
    n : int or None
        Length of each synthetic series. Defaults to len(x).
    alpha : float or None
        AR(1) coefficient. Estimated from x if not provided.
    max_period : float or None
        Maximum period to consider when unmirroring the wavelet results.
        If None, it is set to the maximum possible period based on the
        length of the time series.
    dt : float
        Sampling interval of the time series. Default is 16 days.
    deseason_period : float or None
        Period used for deseasonalisation when estimating the AR(1)
        coefficient.

    Returns
    -------
    dict
        Dictionary containing:
        "alpha"         – the AR(1) coefficient used.
        "wavelet_power" – wavelet power statistics for each time and period.
        "time-averaged" – time-averaged wavelet power statistics for each period.
    """
    # stop and ask the user to provide max_period
    if max_period is None:
        print("Warning: max_period not provided. Setting it to len(x) * dt - dt, Which is the maximum possible period for the wavelet analysis.")
        max_period = len(x) * dt-dt # set it to the maximum possible period, which is the length of the time series in days (len(x) * dt) minus one time step (dt), 
        # this will correspond to the period of a wavelet that has one full cycle in the time series.
    
    # Build the AR(1) ensemble 
    ar1_ensemble = generate_ar1_ensemble(x, n_ensemble=n_ensemble, n=n, alpha=alpha, deseason_period=deseason_period)

    # Resolve alpha so we can store it in the output
    if alpha is None:
        alpha = estimate_alpha(x, deseason_period=deseason_period)

    # Run wavelet analysis on every member 
    wp_list   = []   # one wp DataFrame per ensemble member
    sump_list = []   # one sumpow DataFrame per ensemble member
    
    for i in range(n_ensemble):
        _, _, _, _, _, wp_ar1, _, _, _, _ = run_full_wavelet_analysis(
            ar1_ensemble[i],
            dt=dt, mirror=True, wf='morlet',
            dj=0.1, om0=6, normmean=True,
            mirrormethod=2, period_min=5, period_max=20,
        )

        wp_ar1, sumpow_ar1 = unmirror_wavelet_power(
            ar1_ensemble[i], wp_ar1,
            period_min=32, period_max=max_period,
        )

        wp_list.append(wp_ar1)
        sump_list.append(sumpow_ar1)

    stats = (
        pd.concat(wp_list, ignore_index=True)
          .groupby(['time', 'period'])['rectified_power']
          .pipe(quantile_stats)
          .reset_index()
    )

    stats_sumpow = (stats.groupby(['period'])[['median','p95', 'p25', 'p75']].mean().reset_index())

    # return the results in a dictionary
    return {
        "alpha":         alpha,
        "wavelet_power": stats,
        "time-averaged": stats_sumpow,
    }


In [ ]:
# function to estimate significance threshold for the cross-wavelet power and coherence will be similar 
# but we need to generate two AR(1) ensembles and run a double wavelet analysis instead of a single one. 
# We also need to unmirror the crosspower and coherence instead of the wavelet power.

def estimate_significance_threshold_crosswavelet(x, y, n_ensemble=1000, n=None, dt=16, max_period=None, deseason_period=None, alpha=None):

    """
    Estimate significance thresholds for wavelet power, cros power,
    and coherence using ensembles of AR(1) surrogate time series.

    Two independent AR(1) ensembles are generated, one for each input series.
    A double wavelet analysis is then performed for each pair of surrogate
    series. The wavelet power, crosspower, and coherence are
    calculated and unmirrored to remove the artificial extension introduced
    to reduce edge effects.

    The resulting ensemble distributions are used to calculate quantile
    statistics (median, 25th, 75th, and 95th percentiles) for each time and
    period. Time-averaged statistics are also calculated for each period.

    Parameters
    ----------
    x, y : array-like
        The two input time series.
    n_ensemble : int, optional
        Number of AR(1) surrogate pairs to generate. Default is 1000.
    n : int, optional
        Length of the AR(1) surrogate series.
    dt : float, optional
        Sampling interval of the input time series. Default is 16 days.
    max_period : float, optional
        Maximum period included in the wavelet analysis. If None, it is set
        to the maximum possible period based on the length of the time series.
    deseason_period : float, optional
        Period used for deseasonalisation when estimating the AR(1) model.
    alpha : float, optional
        Alpha parameter used for the AR(1) model. If None, it is estimated
        from the input series.

    Returns
    -------
    results_dictionary : dict
        Dictionary containing the ensemble statistics for wavelet power,
        cross-wavelet power, and coherence, including both time-dependent
        and time-averaged statistics.
    """
    
    if max_period is None:
        print("Warning: max_period not provided. Setting it to len(x) * dt - dt, Which is the maximum possible period for the wavelet analysis.")
        max_period = len(x) * dt-dt # set it to the maximum possible period, which is the length of the time series in days (len(x) * dt) minus one time step (dt), 
        # this will correspond to the period of a wavelet that has one full cycle in the time series.
    
    # Resolve alpha so we can store it in the output
    if alpha is None:
        alpha = estimate_alpha(x, deseason_period=deseason_period)
    
    # Build two AR(1) ensembles 
    ar1_ensemble_x = generate_ar1_ensemble(x, n_ensemble=n_ensemble, n=n, deseason_period=deseason_period)
    ar1_ensemble_y = generate_ar1_ensemble(y, n_ensemble=n_ensemble, n=n, deseason_period=deseason_period)
    
    # run double wavelet analysis on every member of the ensemble and unmirror the crosspower and coherence
    # lists to store results and compute stats later
    wp_x_list   = [] 
    wp_xsump_list = []  
    
    wp_y_list = []  
    wp_ysump_list = []
    
    crosspower_list = [] 
    crosspower_sump_list = []
      
    coherence_list = []   
    coherence_sump_list = []
    
     # Run wavelet analysis on every member 
    for i in range(n_ensemble):
        _, _, _, _, _, wavelet_power_df1, _, _, _, _, _, wavelet_power_df2, xypower_df, phasexy_df, R2ns_df, _, _, _ = run_double_wavelet_analysis(
            ar1_ensemble_x[i], ar1_ensemble_y[i],
            dt=dt, mirror=True, wf='morlet',
            dj=0.1, om0=6, normmean=True,
            mirrormethod=2, period_min=5, period_max=20,
        )
        
        # unmirror
        wp_x, sumpow_wp_x = unmirror_wavelet_power(
            ar1_ensemble_x[i], wavelet_power_df1,
            period_min=32, period_max=max_period,
        )
        wp_y, sumpow_wp_y = unmirror_wavelet_power(
            ar1_ensemble_y[i], wavelet_power_df2,
            period_min=32, period_max=max_period,
        )
        xypower, sumpow_xypower = unmirror_crosspower(
            ar1_ensemble_x[i], xypower_df,
            period_min=32, period_max=max_period,
        )
        phasexy = unmirror_phase(
            ar1_ensemble_x[i],  phasexy_df,
            period_min=32, period_max=max_period,
        )
        R2ns, sumpow_R2ns = unmirror_coherence(
            ar1_ensemble_x[i], R2ns_df,
            period_min=32, period_max=max_period,
        )

        # store results
        wp_x_list.append(wp_x)
        wp_xsump_list.append(sumpow_wp_x)
        wp_y_list.append(wp_y)
        wp_ysump_list.append(sumpow_wp_y)
        crosspower_list.append(xypower)
        crosspower_sump_list.append(sumpow_xypower)
        coherence_list.append(R2ns)
        coherence_sump_list.append(sumpow_R2ns)

    # aggregate across the ensemble
    # power and coherence
    stats_x = (
        pd.concat(wp_x_list, ignore_index=True)
          .groupby(['time', 'period'])['rectified_power']
          .pipe(quantile_stats)
          .reset_index()
    )
    stats_y = (
        pd.concat(wp_y_list, ignore_index=True)
          .groupby(['time', 'period'])['rectified_power']
          .pipe(quantile_stats)
          .reset_index()
    )

    stats_crosspower = (
        pd.concat(crosspower_list, ignore_index=True)
          .groupby(['time', 'period'])['xypower_rectified']
          .pipe(quantile_stats)
          .reset_index()
    )

    stats_coherence = (
        pd.concat(coherence_list, ignore_index=True)
          .groupby(['time', 'period'])['R2ns']
          .pipe(quantile_stats)
          .reset_index()
    )
    
    # # time averaged stats
    sumpow_x_stats = (stats_x.groupby("period")[["median","p95", "p25", "p75"]].mean().reset_index())
    sumpow_y_stats = (stats_y.groupby("period")[["median","p95", "p25", "p75"]].mean().reset_index())
    crosspower_sumpow_stats = (stats_crosspower.groupby("period")[["median","p95", "p25", "p75"]].mean().reset_index())
    coherence_sumpow_stats = (stats_coherence.groupby("period")[["median","p95", "p25", "p75"]].mean().reset_index())
    print("Time-averaged stats computed.")
    

    # return the results in a dictionary
    results_dictionary = {
        "alpha": alpha,
        "wavelet_power_x": stats_x,
        "wavelet_power_y": stats_y,
        "crosspower": stats_crosspower,
        "coherence": stats_coherence,
        "sumpow_x": sumpow_x_stats,
        "sumpow_y": sumpow_y_stats,
        "crosspower_sumpow": crosspower_sumpow_stats,
        "coherence_sumpow": coherence_sumpow_stats}
    
    
    return results_dictionary